# Grupo 16 - Proyecto Data Science
## Modelo supervisado: ¿la acción sube el año siguiente?

**Problema:** clasificación binaria. A partir de los indicadores financiero-contables de una empresa en el año *t*, predecir si su acción **sube (1) o no (0)** durante el año *t+1* (`clase`).

**Datos:** "200+ Financial Indicators of US stocks (2014-2018)", 15 ratios + sector (los mismos del EDA).

### Índice
* **Parte 0.** Preparación del entorno.
* **Parte 1.** Datos: se repiten las decisiones del EDA.
* **Parte 2.** El problema y el piso de referencia.
* **Parte 3.** La partición: por qué tiene que ser temporal.
* **Parte 4.** Preprocesamiento dentro de un `Pipeline`.
* **Parte 5.** Modelos base y validación cruzada.
* **Parte 6.** Overfitting: la curva de validación del árbol.
* **Parte 7.** Optimización de hiperparámetros.
* **Parte 8.** Evaluación final en test (una sola vez).
* **Parte 9.** Controles: ¿el modelo aprendió algo o está adivinando?
* **Parte 10.** Interpretación: ¿en qué se fija el modelo?
* **Parte 11.** Conclusiones.

### Decisiones de diseño (y por qué)

| Decisión | Por qué |
|---|---|
| Partición **temporal**: entrenar con 2014-2017, testear con 2018 | En la práctica se predice el futuro con datos del pasado. Además, el mercado sube o baja casi todo junto en un año, y una partición al azar mezclaría años. |
| Validación cruzada **temporal** (entrenar con años anteriores, validar en el siguiente) | Imita el uso real. La validación cruzada por empresa mezcla años y da resultados demasiado optimistas (lo mostramos en la Parte 5). |
| Métrica principal: **AUC**; también exactitud balanceada | La proporción de acciones que suben cambia mucho según el año (28% a 70%). La exactitud y el F1 de la clase "sube" dependen de esa proporción y pueden engañar. |
| Imputación y recorte de extremos **dentro del `Pipeline`** | Así los valores de relleno y los límites se calculan solo con datos de entrenamiento. |
| Piso, control con etiquetas mezcladas y objetivo alternativo | Para comprobar que el modelo no está solo adivinando. |

## Parte 0. Preparación del entorno

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, balanced_accuracy_score,
                             classification_report, f1_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SEMILLA = 42

## Parte 1. Datos: se repiten las decisiones del EDA

Partimos del archivo unificado que guardó el EDA (`financial_data_unificado.csv`) y repetimos, en una sola celda, las decisiones ya justificadas allí:

1. Nos quedamos con las 15 variables elegidas, más `Ticker`, `Year`, `Sector` y el objetivo, y las traducimos.
2. Los ceros de `pe_ratio` y `pb_ratio` se tratan como faltantes.
3. Se eliminan las filas con 9 o más de los 15 ratios vacíos.

**No imputamos ni recortamos extremos acá.** Eso se hace dentro del `Pipeline` (Parte 4), para que se ajuste solo con el entrenamiento. La única diferencia con el EDA es que ahí la imputación usaba la mediana de cada sector y año; acá usamos la mediana del entrenamiento y agregamos una columna que indica "este dato faltaba".

Además se anulan 15 capitalizaciones iguales a 0 y 5 liquidez corriente negativas, valores imposibles.

In [ ]:
def leer_csv(nombre):
    """Lee un CSV desde /content/ (Colab) y, si no existe, desde el directorio local."""
    try:
        return pd.read_csv("/content/" + nombre)
    except FileNotFoundError:
        return pd.read_csv(nombre)


traduccion = {
    "Ticker": "ticker", "Year": "anio", "Sector": "sector", "Market Cap": "capitalizacion_mercado",
    "PE ratio": "pe_ratio", "PB ratio": "pb_ratio", "ROE": "roe", "returnOnAssets": "roa", "ROIC": "roic",
    "Net Profit Margin": "margen_neto", "Free Cash Flow margin": "margen_flujo_caja_libre",
    "Current ratio": "liquidez_corriente", "quickRatio": "liquidez_inmediata",
    "Debt to Equity": "deuda_patrimonio", "Interest Coverage": "cobertura_intereses",
    "Revenue Growth": "crecimiento_ingresos", "EPS Growth": "crecimiento_eps",
    "Dividend Yield": "rendimiento_dividendo",
    "Price Var Next Year": "variacion_precio_siguiente_anio", "Class": "clase",
}
columnas_ratio = [
    "capitalizacion_mercado", "pe_ratio", "pb_ratio", "roe", "roa", "roic", "margen_neto",
    "margen_flujo_caja_libre", "liquidez_corriente", "liquidez_inmediata", "deuda_patrimonio",
    "cobertura_intereses", "crecimiento_ingresos", "crecimiento_eps", "rendimiento_dividendo",
]

df = leer_csv("financial_data_unificado.csv")
df = df[list(traduccion)].rename(columns=traduccion)

# Decisiones del EDA
df[["pe_ratio", "pb_ratio"]] = df[["pe_ratio", "pb_ratio"]].replace(0, np.nan)
faltantes_en_fila = df[columnas_ratio].isna().sum(axis=1)
df = df[faltantes_en_fila < 9].copy()

# Valores imposibles
df.loc[df["capitalizacion_mercado"] <= 0, "capitalizacion_mercado"] = np.nan
df.loc[df["liquidez_corriente"] < 0, "liquidez_corriente"] = np.nan

# La capitalización tiene valores desde millones hasta billones: usamos su logaritmo
df["log_capitalizacion"] = np.log10(df["capitalizacion_mercado"])

print("Filas:", len(df), "| Empresas:", df["ticker"].nunique())
df.head(3)

## Parte 2. El problema y el piso de referencia

Definimos `y = clase` (1 si la acción sube el año siguiente). Las variables predictoras son los 15 ratios (con la capitalización en logaritmo) y el sector. **Quedan afuera** `ticker` y `anio` (identificadores) y `variacion_precio_siguiente_anio`, que es la respuesta: dejarla en `X` sería hacer trampa.

Antes de modelar miramos cuánto vale el **piso**: lo que se logra sin aprender nada.

In [ ]:
cols_num = ["log_capitalizacion", "pe_ratio", "pb_ratio", "roe", "roa", "roic", "margen_neto",
            "margen_flujo_caja_libre", "liquidez_corriente", "liquidez_inmediata", "deuda_patrimonio",
            "cobertura_intereses", "crecimiento_ingresos", "crecimiento_eps", "rendimiento_dividendo"]
cols_cat = ["sector"]

X = df[cols_num + cols_cat]
y = df["clase"]
grupos = df["ticker"]
anios = df["anio"]

print("Proporción general de acciones que suben:", round(y.mean(), 3))
df.groupby("anio")["clase"].agg(proporcion_sube="mean", empresas="size").round(3)

**Para discutir:** la proporción de acciones que suben va de 28% a 70% según el año. Eso condiciona cómo se evalúa:

* Un modelo que dice "sube" siempre acierta el 68% de las veces en un año como 2018, sin aprender nada. La **exactitud** y el **F1 de la clase "sube"** premian ese comportamiento.
* Por eso usamos como métrica principal el **AUC**, que no depende de la proporción de clases (0,5 = azar), junto con la **exactitud balanceada** (promedio del acierto en cada clase, donde el piso vale 0,5).

## Parte 3. La partición: por qué tiene que ser temporal

Entrenamos con los años **2014 a 2017** y dejamos **2018** como test. Hay tres razones:

1. Se predice el futuro con datos del pasado, y una partición al azar dejaría datos de 2018 en el entrenamiento.
2. El mercado sube o baja casi todo junto en un año. Si mezclamos años, el modelo puede aprender "este año fue bueno" en lugar de "esta empresa es buena".
3. Una misma empresa aparece hasta 5 veces (una por año), con indicadores parecidos.

**Validación cruzada:** dentro del entrenamiento usamos folds **temporales**: entrenar con los años anteriores y validar en el siguiente (2015, 2016 y 2017). En la Parte 5 la comparamos con una validación cruzada agrupada por empresa, la que usan las notebooks de clase, para ver cuánto cambia.

In [ ]:
es_train = (anios <= 2017).values
X_train, X_test = X[es_train], X[~es_train]
y_train, y_test = y[es_train], y[~es_train]
grupos_train = grupos[es_train]
anios_train = anios[es_train].values

print("Entrenamiento (2014-2017):", X_train.shape, "| proporción que sube:", round(y_train.mean(), 3))
print("Test (2018):              ", X_test.shape, "| proporción que sube:", round(y_test.mean(), 3))
print("Empresas del test que también están en el entrenamiento:",
      len(set(grupos[~es_train]) & set(grupos_train)), "de", grupos[~es_train].nunique())

# Folds temporales: entrenar con años anteriores, validar en el año siguiente
cv_temporal = [(np.where(anios_train < a)[0], np.where(anios_train == a)[0]) for a in (2015, 2016, 2017)]
# Folds agrupados por empresa (los de las notebooks de clase), solo para comparar
cv_empresas = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
                   .split(X_train, y_train, grupos_train))

Nota: el test de 2018 tiene 68% de acciones que suben contra 51% del entrenamiento. Es un cambio de proporción esperable entre años, y es una razón más para mirar el AUC.

## Parte 4. Preprocesamiento dentro de un `Pipeline`

Todo lo que "aprende" de los datos (la mediana para rellenar, los límites para recortar, la media para escalar) se calcula **solo con el entrenamiento** y se aplica después al test. Para eso va dentro del `Pipeline`.

* **Imputación** con la mediana, más una columna indicadora "el dato faltaba" por cada ratio con faltantes. Que falte el P/E puede ser informativo: probablemente la empresa no tiene ganancias positivas.
* **Recorte de extremos** (percentiles 1 y 99) y **escalado**: solo para la regresión logística, que es sensible a los valores extremos y a la escala. Los árboles no los necesitan.
* **Sector** con *one hot*.

El recorte no existe en scikit-learn, así que escribimos una pequeña clase `Winsorizador`. Aprende los límites en `fit` y recorta en `transform`.

In [ ]:
class Winsorizador(BaseEstimator, TransformerMixin):
    """Recorta cada columna a sus percentiles bajo y alto (calculados con los datos de entrenamiento)."""

    def __init__(self, bajo=0.01, alto=0.99):
        self.bajo = bajo
        self.alto = alto

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        X = np.asarray(X, dtype=float)
        self.lim_bajo_ = np.nanquantile(X, self.bajo, axis=0)
        self.lim_alto_ = np.nanquantile(X, self.alto, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lim_bajo_, self.lim_alto_)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_ if input_features is None else np.asarray(input_features, dtype=object)


def armar_preprocesador(lineal):
    pasos = [("recorte", Winsorizador())] if lineal else []
    pasos.append(("imputador", SimpleImputer(strategy="median", add_indicator=True)))
    if lineal:
        pasos.append(("escalador", StandardScaler()))
    return ColumnTransformer(
        [("num", Pipeline(pasos), cols_num),
         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols_cat)],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")


def armar_modelo(clasificador, lineal=False):
    return Pipeline([("prep", armar_preprocesador(lineal)), ("clf", clasificador)])


# Comprobación: el preprocesador ajustado con el entrenamiento
prep_prueba = armar_preprocesador(lineal=False).fit(X_train)
print("Columnas que recibe el modelo:", prep_prueba.transform(X_train).shape[1])
print("Indicadores de dato faltante:", [c for c in prep_prueba.get_feature_names_out() if c.startswith("missingindicator")])

## Parte 5. Modelos base y validación cruzada

Comparamos cuatro modelos, como en la Notebook 21: el **piso**, la regresión logística, un árbol de profundidad 4 y un Random Forest. Se comparan **solo con validación cruzada sobre el entrenamiento**: el test no se toca todavía.

Medimos AUC, exactitud balanceada y F1 con los folds temporales, y el AUC también con los folds agrupados por empresa para ver la diferencia.

In [ ]:
modelos = {
    "Piso (Dummy)": armar_modelo(DummyClassifier(strategy="prior")),
    "Regresión logística": armar_modelo(LogisticRegression(max_iter=1000), lineal=True),
    "Árbol (profundidad 4)": armar_modelo(DecisionTreeClassifier(max_depth=4, random_state=SEMILLA)),
    "Random Forest": armar_modelo(RandomForestClassifier(n_estimators=100, min_samples_leaf=5,
                                                          random_state=SEMILLA, n_jobs=-1)),
}

def media_desvio(r, clave):
    return f"{r[clave].mean():.3f} ± {r[clave].std():.3f}"

filas = []
for nombre, modelo in modelos.items():
    r_t = cross_validate(modelo, X_train, y_train, cv=cv_temporal,
                         scoring=["roc_auc", "balanced_accuracy", "f1"])
    r_e = cross_validate(modelo, X_train, y_train, cv=cv_empresas, scoring=["roc_auc"])
    filas.append({"Modelo": nombre,
                  "AUC (cv temporal)": media_desvio(r_t, "test_roc_auc"),
                  "Exactitud balanceada (cv temporal)": media_desvio(r_t, "test_balanced_accuracy"),
                  "F1 (cv temporal)": media_desvio(r_t, "test_f1"),
                  "AUC (cv por empresa)": media_desvio(r_e, "test_roc_auc")})
pd.DataFrame(filas).set_index("Modelo")

**Para discutir:**

* **La validación cruzada por empresa da un AUC más alto que la temporal** (comparen las dos columnas de AUC). No es que el modelo sea mejor: ese método mezcla años en cada fold y le permite aprovechar que el mercado se movió distinto cada año. La temporal es la estimación honesta de lo que se puede hacer hacia adelante, y es la que usamos para decidir.
* Todos los modelos superan al piso (AUC 0,5), pero **por poco** y con desvíos comparables a las diferencias entre modelos. Con esta información, ¿podemos decir cuál es el mejor?
* El F1 del piso es engañoso (su valor depende de la proporción de clases del fold), así que no lo usamos para comparar.

## Parte 6. Overfitting: la curva de validación del árbol

Como en la Notebook 22, vemos cómo cambia el AUC de un árbol al dejarlo crecer. La brecha entre el AUC de entrenamiento y el de validación temporal es el overfitting.

In [ ]:
profundidades = [1, 2, 3, 4, 6, 8, 12]
auc_tr, auc_va = [], []
for d in profundidades:
    arbol = armar_modelo(DecisionTreeClassifier(max_depth=d, random_state=SEMILLA))
    r = cross_validate(arbol, X_train, y_train, cv=cv_temporal, scoring="roc_auc", return_train_score=True)
    auc_tr.append(r["train_score"].mean())
    auc_va.append(r["test_score"].mean())

plt.plot(profundidades, auc_tr, "o-", label="Entrenamiento", color="#C44E52")
plt.plot(profundidades, auc_va, "o-", label="Validación cruzada temporal", color="#4C72B0")
plt.axhline(0.5, color="gray", linestyle="--", label="Azar")
plt.xlabel("Profundidad máxima del árbol")
plt.ylabel("AUC")
plt.title("Curva de validación del árbol de decisión")
plt.legend()
plt.tight_layout()
plt.show()

A medida que el árbol crece, el AUC de entrenamiento sube mucho (el árbol memoriza), pero el de validación **no mejora, y empeora**. La señal de los ratios es débil y ruidosa, así que los modelos simples funcionan mejor. Esto anticipa el resultado de la búsqueda de hiperparámetros.

## Parte 7. Optimización de hiperparámetros

Elegimos los dos modelos más prometedores y buscamos sus hiperparámetros con `GridSearchCV`, con la **validación cruzada temporal** y el **AUC** como métrica, solo sobre el entrenamiento.

* **Random Forest:** cantidad de árboles, profundidad máxima y mínimo de muestras por hoja.
* **Regresión logística:** la fuerza de la regularización `C` (valores chicos regularizan más).

In [ ]:
grilla_rf = {
    "clf__n_estimators": [100, 300],
    "clf__max_depth": [3, 5, 10, None],
    "clf__min_samples_leaf": [5, 20, 50],
}
busqueda_rf = GridSearchCV(
    armar_modelo(RandomForestClassifier(random_state=SEMILLA, n_jobs=-1)),
    param_grid=grilla_rf, cv=cv_temporal, scoring="roc_auc", n_jobs=1)
inicio = time.time()
busqueda_rf.fit(X_train, y_train)
print(f"Random Forest ({time.time() - inicio:.0f} segundos)")
print("  Mejores hiperparámetros:", busqueda_rf.best_params_)
print(f"  AUC en validación cruzada temporal: {busqueda_rf.best_score_:.3f}")

grilla_lr = {"clf__C": [0.001, 0.01, 0.1, 1, 10]}
busqueda_lr = GridSearchCV(
    armar_modelo(LogisticRegression(max_iter=1000), lineal=True),
    param_grid=grilla_lr, cv=cv_temporal, scoring="roc_auc")
busqueda_lr.fit(X_train, y_train)
print("Regresión logística")
print("  Mejores hiperparámetros:", busqueda_lr.best_params_)
print(f"  AUC en validación cruzada temporal: {busqueda_lr.best_score_:.3f}")

modelo_rf = busqueda_rf.best_estimator_
modelo_lr = busqueda_lr.best_estimator_

**Para discutir:** la búsqueda prefiere un Random Forest con árboles poco profundos (se queda con la profundidad más baja de la grilla) y una regresión logística muy regularizada. Los dos llegan a un AUC parecido y levemente mayor que sin ajustar. Es consistente con lo anterior: la señal es débil y gana la simplicidad. Como los dos empatan en validación, los llevamos a test como **finalistas**.

## Parte 8. Evaluación final en test (una sola vez)

Usamos 2018 **una sola vez**, con los modelos ya elegidos y reentrenados con todo el entrenamiento (`best_estimator_`). Incluimos el piso como referencia.

In [ ]:
def metricas_test(modelo, nombre):
    proba = modelo.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {"Modelo": nombre,
            "AUC": roc_auc_score(y_test, proba),
            "Exactitud balanceada": balanced_accuracy_score(y_test, pred),
            "Exactitud": (pred == y_test).mean(),
            "F1 (clase 'sube')": f1_score(y_test, pred),
            "% que predice 'sube'": pred.mean()}

piso = armar_modelo(DummyClassifier(strategy="prior")).fit(X_train, y_train)
tabla_test = pd.DataFrame([
    metricas_test(piso, "Piso (Dummy)"),
    metricas_test(modelo_lr, "Regresión logística (ajustada)"),
    metricas_test(modelo_rf, "Random Forest (ajustado)"),
]).set_index("Modelo").round(3)
print("Proporción real de acciones que suben en el test:", round(y_test.mean(), 3))
tabla_test

In [ ]:
proba_rf = modelo_rf.predict_proba(X_test)[:, 1]
pred_rf = (proba_rf >= 0.5).astype(int)
print(classification_report(y_test, pred_rf, target_names=["No sube", "Sube"]))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_rf, display_labels=["No sube", "Sube"],
                                        cmap="Blues", ax=ejes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, proba_rf, ax=ejes[1], name="Random Forest")
RocCurveDisplay.from_predictions(y_test, modelo_lr.predict_proba(X_test)[:, 1], ax=ejes[1],
                                 name="Regresión logística")
ejes[1].plot([0, 1], [0, 1], "--", color="gray")
ejes[0].set_title("Matriz de confusión (Random Forest)")
ejes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

**Cómo leer la tabla:**

* El AUC de los dos modelos es claramente mayor que 0,5 y la exactitud balanceada supera al piso (0,5): **el modelo distingue algo entre las empresas que suben y las que no**.
* La **exactitud casi no mejora** respecto del piso y el **F1 del piso es mayor** que el de los modelos. No es una contradicción: en 2018 sube el 68% de las acciones y el piso (que dice "sube" siempre) acierta esa clase por construcción. Los modelos, en cambio, se equivocan a propósito en algunas para distinguir mejor. Por eso la exactitud y el F1 de la clase "sube" no sirven para evaluar acá, y usamos el AUC y la exactitud balanceada.
* El modelo predice "sube" menos veces que la proporción real: aprendió con 51% de subas y en el test hay 68%. Un AUC no se ve afectado por ese desfase, pero la exactitud sí.

## Parte 9. Controles: ¿el modelo aprendió algo o está adivinando?

Tres comprobaciones para no engañarnos.

**Control 1. Etiquetas mezcladas.** Se mezclan al azar las etiquetas del entrenamiento y se reentrena. Ahí **no hay nada que aprender**, así que el AUC en test debería rondar 0,5. Si diera bastante más, habría una fuga de información en el proceso.

In [ ]:
rng = np.random.RandomState(SEMILLA)
aucs_mezcladas = []
for _ in range(5):
    y_mezclada = pd.Series(rng.permutation(y_train.values), index=y_train.index)
    modelo = clone(modelo_rf).fit(X_train, y_mezclada)
    aucs_mezcladas.append(roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]))

print("AUC en test con etiquetas mezcladas (5 repeticiones):", np.round(aucs_mezcladas, 3))
print("AUC en test con etiquetas reales:", round(roc_auc_score(y_test, proba_rf), 3))

Con etiquetas mezcladas el AUC oscila entre 0,39 y 0,56: son valores de azar (con 4.140 filas del mismo año, el resultado se mueve bastante de una repetición a otra), todos muy por debajo del 0,73 con etiquetas reales. El proceso no filtra información del test.

**Control 2. ¿Es estable el resultado entre años?** Un solo año de test puede ser favorable o desfavorable por casualidad. Para ver cuánto varía, calculamos el AUC **dentro de cada año**, entrenando con los otros cuatro. Es un control descriptivo (usa años futuros para entrenar), no una estimación de uso real.

In [ ]:
resultados_anio = {}
for nombre, modelo in [("Regresión logística", modelo_lr), ("Random Forest", modelo_rf)]:
    aucs = []
    for a in range(2014, 2019):
        m = clone(modelo).fit(X[anios != a], y[anios != a])
        aucs.append(roc_auc_score(y[anios == a], m.predict_proba(X[anios == a])[:, 1]))
    resultados_anio[nombre] = aucs

auc_por_anio = pd.DataFrame(resultados_anio, index=range(2014, 2019))
ax = auc_por_anio.plot(kind="bar", rot=0, color=["#55A868", "#4C72B0"])
ax.axhline(0.5, color="gray", linestyle="--")
ax.set_ylabel("AUC dentro del año")
ax.set_xlabel("año de análisis")
ax.set_title("AUC por año (entrenando con los otros cuatro años)")
plt.tight_layout()
plt.show()
auc_por_anio.round(3)

El AUC cambia bastante de un año a otro: 2018 es el año donde los ratios separan mejor, y 2017 el peor. Hay que tener cuidado: **el resultado en el test de 2018 es el más favorable de los cinco años**, y no conviene presentarlo como el desempeño esperado en cualquier año.

**Control 3. Un objetivo que no depende del mercado.** Cambiamos la variable a predecir: en lugar de "sube", **"sube más que la mediana de su año"**. Así se elimina el efecto del mercado y las clases quedan 50/50. Es un problema más difícil, y sirve para ver cuánto del resultado depende de qué año fue bueno o malo.

In [ ]:
mediana_anio = df.groupby("anio")["variacion_precio_siguiente_anio"].transform("median")
y_rel = (df["variacion_precio_siguiente_anio"] > mediana_anio).astype(int)
y_rel_train, y_rel_test = y_rel[es_train], y_rel[~es_train]

filas = []
for nombre, modelo in [("Regresión logística", modelo_lr), ("Random Forest", modelo_rf)]:
    r = cross_validate(clone(modelo), X_train, y_rel_train, cv=cv_temporal, scoring="roc_auc")
    m = clone(modelo).fit(X_train, y_rel_train)
    filas.append({"Modelo": nombre,
                  "AUC cv temporal": r["test_score"].mean(),
                  "AUC test 2018": roc_auc_score(y_rel_test, m.predict_proba(X_test)[:, 1])})
print("Proporción que 'sube más que la mediana' en el test:", round(y_rel_test.mean(), 3))
pd.DataFrame(filas).set_index("Modelo").round(3)

Con este objetivo el AUC baja y queda más cerca del de la validación cruzada temporal. Una parte del resultado de la Parte 8 viene entonces de que 2018 fue un año particular, y la capacidad real de los ratios para separar ganadoras de perdedoras dentro de un mismo año es **modesta**.

## Parte 10. Interpretación: ¿en qué se fija el modelo?

La interpretación solo tiene sentido porque el modelo supera al azar (Partes 8 y 9). Aun así, la señal es débil, así que las importancias hay que leerlas con cuidado, y **no prueban causalidad**.

### Coeficientes de la regresión logística

Las variables están estandarizadas, así que los coeficientes son comparables. Un coeficiente positivo empuja hacia "sube" y uno negativo hacia "no sube". Las columnas `missingindicator_...` indican que el dato original faltaba.

In [ ]:
regresion = modelo_lr.named_steps["clf"]
nombres_lr = modelo_lr.named_steps["prep"].get_feature_names_out()
coeficientes = pd.Series(regresion.coef_[0], index=nombres_lr).sort_values()
extremos = pd.concat([coeficientes.head(6), coeficientes.tail(6)])

extremos.plot(kind="barh", color=["#C44E52" if v < 0 else "#55A868" for v in extremos])
plt.axvline(0, color="gray", linewidth=1)
plt.xlabel("Coeficiente (variables estandarizadas)")
plt.title("Regresión logística: variables con más peso")
plt.tight_layout()
plt.show()

### Importancia por permutación (Random Forest, test)

Se mezclan al azar los valores de una variable en el test y se mide cuánto cae el AUC. Si no cae, el modelo no la usa. Funciona con cualquier modelo.

In [ ]:
importancia = permutation_importance(modelo_rf, X_test, y_test, scoring="roc_auc",
                                     n_repeats=10, random_state=SEMILLA)
serie = pd.Series(importancia.importances_mean, index=X_test.columns).sort_values().tail(10)
error = pd.Series(importancia.importances_std, index=X_test.columns)[serie.index]

serie.plot(kind="barh", xerr=error, color="#4C72B0")
plt.xlabel("Caída del AUC al mezclar la variable")
plt.title("Importancia por permutación (Random Forest, test)")
plt.tight_layout()
plt.show()

**Para discutir:**

* Las caídas de AUC son **muy chicas** (la mayor, la del sector, es de apenas 0,014; el resto está entre 0,002 y 0,007): ninguna variable por sí sola explica el resultado, y eso coincide con las correlaciones débiles del EDA.
* Aparecen el **sector**, la rentabilidad (`margen_neto`, `roic`) y el tamaño (`log_capitalizacion`). Cuidado con el sector: dentro del entrenamiento, cada sector tuvo años muy distintos (por ejemplo Energy), y el modelo puede estar aprendiendo "cómo le fue a ese sector en esos años" más que algo estable.
* En la regresión logística, el indicador de **P/E faltante** tiene el coeficiente más negativo (alrededor de -0,13), seguido por el sector Energy; el tamaño y el sector Technology empujan hacia "sube". Esto justifica haber agregado indicadores de dato faltante: que falte el P/E (probablemente por ganancias no positivas) informa algo.

### SHAP (opcional)

Como en la Notebook 21, SHAP reparte la predicción entre las variables y muestra el sentido del efecto. Requiere la librería `shap` (Colab la instala con la celda siguiente).

In [ ]:
try:
    import shap
except ImportError:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
        import shap
    except Exception as error_shap:
        print("No se pudo instalar shap:", error_shap)
        shap = None

if shap is not None:
    rf = modelo_rf.named_steps["clf"]
    X_test_t = modelo_rf.named_steps["prep"].transform(X_test).sample(1000, random_state=SEMILLA)
    valores_shap = shap.TreeExplainer(rf).shap_values(X_test_t)
    if isinstance(valores_shap, list):          # según la versión de shap
        valores_shap = valores_shap[1]
    elif valores_shap.ndim == 3:
        valores_shap = valores_shap[:, :, 1]
    shap.summary_plot(valores_shap, X_test_t, show=False, max_display=10)
    plt.title("SHAP: impacto de cada variable en la predicción de 'sube'")
    plt.tight_layout()
    plt.show()

## Parte 11. Conclusiones

1. **Se puede predecir algo, pero poco.** Con la validación temporal, el AUC ronda 0,58-0,59 con los modelos ajustados (azar = 0,5). En el test de 2018 llega a 0,73 con los dos finalistas (regresión logística y Random Forest).
2. **El resultado de 2018 es optimista.** Dentro de cada año el AUC va de 0,51 (2017) a 0,73 (2018), y con el objetivo "sube más que la mediana del año" el AUC en test baja a 0,63-0,64. Una parte de lo que predice el modelo es "cómo le fue al mercado".
3. **Los modelos simples ganan.** Las búsquedas eligieron un Random Forest de árboles poco profundos y una regresión logística muy regularizada; el árbol profundo hace overfitting.
4. **La validación importa.** Con validación cruzada por empresa, el Random Forest parece mucho mejor (AUC 0,65) que con validación temporal (0,57). El primer número es una ilusión.
5. **Exactitud y F1 no sirven para comparar acá**: el piso "siempre sube" acierta 68% en 2018. Usamos AUC y exactitud balanceada.
6. **Variables:** el sector, la rentabilidad, el tamaño y el hecho de que falte el P/E son las que más pesan, aunque cada una aporta muy poco. No hay evidencia de causalidad.

**Limitaciones:** solo 5 años (y un solo test), un único mercado, ratios de un solo año, sin variables macro, y el sector puede estar capturando el comportamiento de cada sector en esos años.

**Posibles extensiones:** definir el objetivo de otra manera (por ejemplo, pérdida neta el año siguiente), probar Gradient Boosting, agregar variaciones de los ratios respecto del año anterior, y usar los perfiles del clustering como variable adicional.